6) SQL 쿼리 생성기

In [ ]:
# [목적] 자연어 기반 SQL 생성에 필요한 모델 환경과 SQLite 데이터베이스 연결을 준비합니다.
# API 환경 변수와 LangSmith 추적을 설정하고 finance.db의 종류와 사용 가능한 테이블을 확인합니다.
# 준비된 db 객체는 이후 SQL 생성 체인과 실행 도구가 데이터 구조를 파악하는 데 사용합니다.
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
from langchain_teddynote import logging
from dotenv import load_dotenv

# .env 파일에 저장된 OpenAI·LangSmith API 키 등의 환경 변수를 불러옵니다.
load_dotenv()
# 이후 LangChain 실행 과정을 LangSmith의 'SQL' 프로젝트에서 추적합니다.
logging.langsmith("SQL")

# SQLDatabase는 데이터베이스 연결과 테이블 구조 정보를 LangChain에 제공하는 객체입니다.
db = SQLDatabase.from_uri("sqlite:///data/finance.db")

print(db.dialect)

print(db.get_usable_table_names())

In [ ]:
# [목적] 사용자의 자연어 질문을 데이터베이스에 맞는 SQL로 바꾸는 체인을 만듭니다.
# ChatOpenAI 모델과 db의 테이블 구조를 연결해 질문에 필요한 쿼리문을 생성하도록 구성합니다.
# 이 단계는 SQL만 작성하며 실제 데이터베이스 실행은 뒤의 도구가 담당합니다.
# temperature=0은 SQL 생성의 무작위성을 줄여 결과를 일관되게 만듭니다.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# create_sql_query_chain은 질문과 DB 스키마를 모델에 전달하는 SQL 생성 전용 체인입니다.
chain = create_sql_query_chain(llm, db)

In [ ]:
# [목적] 고객 이름을 요청하는 자연어 질문으로 SQL 쿼리를 생성하고 내용을 확인합니다.
# question을 체인에 전달하면 모델이 데이터베이스 구조에 맞는 SQL 문자열을 반환합니다.
# 생성된 쿼리는 다음 셀에서 실제 SQLite 데이터베이스에 실행합니다.
generated_sql_query = chain.invoke({"question": "고객의 이름을 나열하세요"})

# repr 형태로 출력하면 줄바꿈이나 따옴표까지 포함한 정확한 쿼리 문자열을 확인할 수 있습니다.
print(generated_sql_query.__repr__())  # 생성된 쿼리를 출력

In [ ]:
# [목적] 모델이 만든 SQL 쿼리를 SQLite 데이터베이스에 실행해 조회 결과를 얻습니다.
# QuerySQLDataBaseTool에 db를 연결하고 generated_sql_query를 query 입력으로 전달합니다.
# 반환된 데이터는 SQL이 올바른지 확인하거나 자연어 답변을 만드는 데 사용할 수 있습니다.
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

# QuerySQLDataBaseTool은 전달받은 SQL을 연결된 DB에서 실행하는 LangChain 도구입니다.
execute_query = QuerySQLDataBaseTool(db=db)  # 쿼리문을 실행하는 도구 생성

execute_query.invoke({"query": generated_sql_query})

In [ ]:
# [목적] SQL 생성과 데이터베이스 실행을 하나의 연속된 체인으로 연결합니다.
# 자연어 질문을 write_query가 SQL로 바꾸고, 그 결과를 execute_query가 즉시 실행합니다.
# 파이프(|) 연결로 두 단계를 한 번의 invoke 호출로 처리할 수 있습니다.
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

execute_query = QuerySQLDataBaseTool(db=db)  # SQL 실행 도구

write_query = create_sql_query_chain(llm, db)  # SQL 쿼리 생성 체인

# 앞 단계의 SQL 문자열이 다음 실행 도구의 query 입력으로 전달됩니다.
chain = write_query | execute_query

In [ ]:
# [목적] SQL 생성·실행 체인으로 특정 고객의 이메일을 한 번에 조회합니다.
# 자연어 질문이 SQL로 변환된 뒤 데이터베이스에서 실행되어 조회 결과가 반환됩니다.
# 결과가 아직 자연어 문장은 아니므로 다음 단계에서 답변 생성 체인을 추가합니다.
chain.invoke({"question": "테디의 이메일을 조회하세요"})

In [ ]:
# [목적] 사용자 질문, 생성된 SQL, 조회 결과를 바탕으로 자연어 답변을 만드는 프롬프트를 정의합니다.
# PromptTemplate의 세 입력값을 하나의 지시문으로 구성해 모델이 조회 결과를 문장으로 설명하게 합니다.
# RunnablePassthrough와 itemgetter는 뒤에서 질문을 유지하며 query와 result를 추가하는 데 사용합니다.
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 중괄호로 표시된 question, query, result는 체인 실행 시 실제 값으로 채워집니다.
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer
the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer:"""
)

In [ ]:
# [목적] SQL 조회 정보를 읽기 쉬운 자연어 답변으로 변환하는 체인을 만듭니다.
# answer_prompt가 입력을 구성하고 모델이 답변을 생성한 뒤 StrOutputParser가 일반 문자열로 바꿉니다.
# 완성된 answer 체인은 다음 셀의 SQL 생성·실행 흐름 마지막에 연결됩니다.
answer = answer_prompt | llm | StrOutputParser()

In [ ]:
# [목적] 질문부터 SQL 생성, 실행, 자연어 답변까지 처리하는 전체 체인을 구성합니다.
# 원래 question을 유지하면서 query와 result를 차례로 추가한 뒤 answer 체인에 모두 전달합니다.
# 한 번의 invoke로 데이터 조회와 사용자용 답변 생성을 끝내기 위한 구조입니다.
chain = (
    # assign은 기존 입력 딕셔너리를 유지하면서 새 query 키를 추가합니다.
    RunnablePassthrough.assign(query=write_query).assign(
        # itemgetter가 생성된 query만 꺼내 SQL 실행 도구에 전달합니다.
        result=itemgetter("query") | execute_query
    )
    | answer
)

In [ ]:
# [목적] 전체 SQL 답변 체인으로 특정 고객의 거래 합계를 계산해 자연어로 답변합니다.
# 질문이 SQL 작성, 데이터베이스 실행, 답변 생성 단계를 순서대로 통과합니다.
# 최종 반환값은 사용자가 바로 읽을 수 있는 거래 합계 설명입니다.
chain.invoke({"question": "테디의 transaction의 합계를 구하세요"})

In [ ]:
# [목적] SQL 도구를 스스로 선택하고 반복 사용할 수 있는 LangChain SQL Agent를 준비합니다.
# 모델과 데이터베이스를 연결해 Agent가 질문 분석, 테이블 확인, SQL 실행 과정을 상황에 맞게 결정하도록 합니다.
# 앞선 고정 체인보다 여러 조회와 비교가 필요한 복합 질문을 처리하는 데 적합합니다.
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

db = SQLDatabase.from_uri("sqlite:///data/finance.db")

# create_sql_agent는 DB 탐색과 쿼리 실행 도구를 모델이 선택할 수 있는 실행기를 만듭니다.
agent_executor = create_sql_agent(
    llm,
    db=db,
    # openai-tools 방식으로 모델이 필요한 SQL 도구와 입력값을 결정합니다.
    agent_type="openai-tools",
    # verbose=True는 Agent의 판단과 도구 실행 과정을 화면에 표시합니다.
    verbose=True
)

In [ ]:
# [목적] SQL Agent로 두 고객의 거래 합계를 각각 구하고 비교하는 복합 질문을 처리합니다.
# Agent가 필요한 조회 순서를 판단하고 SQL 도구를 실행한 뒤 비교 결과를 하나의 답변으로 정리합니다.
# invoke 결과에는 최종 답변과 Agent 실행 결과가 포함됩니다.
agent_executor.invoke(
    {"input": "테디와 설리의 transaction의 합계를 구하고 비교하세요"}
)